In [ ]:
import sys
sys.path.append('..')
import mesh, inflation, numpy as np, importlib, fd_validation, visualization, parametric_pillows, wall_generation
from numpy.linalg import norm

In [ ]:
m, fuseMarkers = wall_generation.triangulate_channel_walls(*parametric_pillows.concentricCircles(8, 50), 0.001)
isheet = inflation.InflatableSheet(m, np.array(fuseMarkers) != 0)

import py_newton_optimizer
opts = py_newton_optimizer.NewtonOptimizerOptions()
opts.useIdentityMetric = True
opts.beta = 1e-4
opts.gradTol = 1e-10

from tri_mesh_viewer import TriMeshViewer
viewer = TriMeshViewer(isheet.visualizationMesh(), width=768, height=640)
viewer.showWireframe()
viewer.show()

In [ ]:
targetSurf = mesh.Mesh('data/pringle.obj')
targetAttractedSheet = inflation.TargetAttractedInflation(isheet, targetSurf)

In [ ]:
targetAttractedSheet.fittingWeight = 0.1

In [ ]:
import time
inflation.benchmark_reset()
isheet.setUseTensionFieldEnergy(True)
isheet.setUseHessianProjectedEnergy(False)
niter = 5000
iterations_per_output = 10
opts.niter = iterations_per_output
isheet.pressure = 30
for step in range(int(niter / iterations_per_output)):
    cr = inflation.inflation_newton(targetAttractedSheet, [], opts)
    # cr = inflation.inflation_newton(isheet, isheet.rigidMotionPinVars, opts)
    viewer.update(False, isheet.visualizationMesh())
    time.sleep(0.01) # Allow some mesh synchronization time for pythreejs
    if cr.numIters() < iterations_per_output: break
inflation.benchmark_report()

In [ ]:
targetAttractedSheet.energy(targetAttractedSheet.EnergyType.Fitting)

In [ ]:
norm(targetAttractedSheet.gradient(targetAttractedSheet.EnergyType.Simulation))

In [ ]:
norm(targetAttractedSheet.gradient(targetAttractedSheet.EnergyType.Fitting))

In [ ]:
fd_validation.validateGrad(targetAttractedSheet, fd_eps=1e-8, etype=targetAttractedSheet.EnergyType.Simulation)

In [ ]:
fd_validation.validateGrad(isheet, fd_eps=1e-9, etype=isheet.EnergyType.Full)

In [ ]:
from numpy.linalg import norm
err, fd_delta_grad, an_delta_grad = fd_validation.validateHessian(targetAttractedSheet, fd_eps=1e-8, etype=targetAttractedSheet.EnergyType.Full)
# Test only the equilibrium rows of the Hessian
# an_delta_grad[targetAttractedSheet.numEquilibriumVars():] = 0.0
# fd_delta_grad[targetAttractedSheet.numEquilibriumVars():] = 0.0
print(norm(an_delta_grad - fd_delta_grad) / norm(fd_delta_grad))
print(an_delta_grad)
print(fd_delta_grad)